##8 - Cricket Winning Probability Analysis




### Step 1: Data Loading



In [ ]:
import pandas as pd

# Load the dataset. Replace 'cricket_matches.csv' with your actual data file path.
# For demonstration, we'll create a dummy DataFrame if the file doesn't exist.
try:
    df = pd.read_csv('cricket_matches.csv')
    print("Dataset loaded from 'cricket_matches.csv'")
except FileNotFoundError:
    print("cricket_matches.csv not found. Creating a dummy dataset for demonstration.")
    data = {
        'team1': ['India', 'Australia', 'England', 'India', 'Australia'],
        'team2': ['Pakistan', 'England', 'New Zealand', 'Sri Lanka', 'South Africa'],
        'venue': ['Mumbai', 'Sydney', 'London', 'Colombo', 'Melbourne'],
        'toss_winner': ['India', 'England', 'England', 'India', 'Australia'],
        'match_winner': ['India', 'England', 'England', 'India', 'Australia'],
        'runs_scored_team1': [180, 250, 160, 200, 220],
        'wickets_lost_team1': [5, 7, 8, 4, 6],
        'runs_scored_team2': [170, 245, 165, 180, 215],
        'wickets_lost_team2': [8, 9, 6, 7, 7],
        'match_date': ['2023-01-01', '2023-01-05', '2023-01-10', '2023-01-15', '2023-01-20']
    }
    df = pd.DataFrame(data)

# Display the first few rows of the DataFrame
display(df.head())

### Step 2: Feature Engineering



In [ ]:
from sklearn.preprocessing import LabelEncoder

# Convert categorical features into numerical representations
le = LabelEncoder()
df['team1_encoded'] = le.fit_transform(df['team1'])
df['team2_encoded'] = le.fit_transform(df['team2'])
df['venue_encoded'] = le.fit_transform(df['venue'])
df['toss_winner_encoded'] = le.fit_transform(df['toss_winner'])

# Create a target variable: 1 if team1 wins, 0 if team2 wins
df['target'] = df.apply(lambda row: 1 if row['match_winner'] == row['team1'] else 0, axis=1)

# Example of a simple derived feature: home advantage (if venue matches team1's 'home' ground, which is simplistic here)
df['is_team1_home'] = df.apply(lambda row: 1 if row['team1'] == 'India' and row['venue'] == 'Mumbai' else 0, axis=1)

# Display the DataFrame with new features
display(df.head())

### Step 3: Model Training



In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Define features (X) and target (y)
features = ['team1_encoded', 'team2_encoded', 'venue_encoded', 'toss_winner_encoded', 'runs_scored_team1', 'wickets_lost_team1', 'runs_scored_team2', 'wickets_lost_team2', 'is_team1_home']
X = df[features]
y = df['target']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train the Logistic Regression model
model = LogisticRegression(solver='liblinear') # 'liblinear' is good for small datasets
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

### Step 4: Prediction



In [ ]:
import numpy as np

# Example of a new match to predict
# Let's assume a match between India (team1) and Pakistan (team2) at Mumbai.
# India wins the toss.
# Team 1 scores 190/6, Team 2 scores 185/7

# You would need to encode these values similar to how the training data was encoded
# For simplicity, we'll use example encoded values based on our dummy data's encoding logic.
# In a real scenario, you'd use the same LabelEncoder objects fitted earlier.

# Example: India encoded as 1, Pakistan as 2, Mumbai as 1
new_match_data = pd.DataFrame({
    'team1_encoded': [1], # India
    'team2_encoded': [2], # Pakistan
    'venue_encoded': [1], # Mumbai
    'toss_winner_encoded': [1], # India
    'runs_scored_team1': [190],
    'wickets_lost_team1': [6],
    'runs_scored_team2': [185],
    'wickets_lost_team2': [7],
    'is_team1_home': [1] # India playing at Mumbai
})

# Predict the probability of team1 winning
# predict_proba returns probabilities for both classes (0 and 1)
winning_probabilities = model.predict_proba(new_match_data[features])

# The probability of team1 winning is the second column (index 1)
prob_team1_wins = winning_probabilities[0][1]
prob_team2_wins = winning_probabilities[0][0]

print(f"Probability of Team 1 (India) winning: {prob_team1_wins:.2f}")
print(f"Probability of Team 2 (Pakistan) winning: {prob_team2_wins:.2f}")

if prob_team1_wins > prob_team2_wins:
    print(f"Based on the model, Team 1 (India) is more likely to win.")
else:
    print(f"Based on the model, Team 2 (Pakistan) is more likely to win.")